# CHARTER_OPEN_METEO_STREAM_SMOKE
**Layer A** smoke on [`MWasil/Open_Meteo_weather_dataset_from_2024_to_2025`](https://huggingface.co/datasets/MWasil/Open_Meteo_weather_dataset_from_2024_to_2025) (`train`).
Statistic: correlation between row order and `temperature_2m`; null: **y-shuffle** on temperatures (energy-relevant tabular stream; not a forecast claim).


In [ ]:
from methodology_preamble import init_notebook, build_run_card, assert_run_card
from datasets import load_dataset
import numpy as np

info = init_notebook()
caps = info["methodology_caps"]["permutation_test"]
n_perm = max(39, min(149, int(caps["n_perm_max"])))

rows = list(
    load_dataset(
        "MWasil/Open_Meteo_weather_dataset_from_2024_to_2025",
        split="train",
        streaming=True,
    ).take(80)
)
y = np.array([float(r["temperature_2m"]) for r in rows], dtype=float)
y = y[np.isfinite(y)]
x = np.arange(len(y), dtype=float)


def _y_shuffle(arr: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    a = np.asarray(arr).ravel()
    idx = rng.permutation(a.size)
    return a[idx].copy()


def _corr(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.corrcoef(np.asarray(a).ravel(), np.asarray(b).ravel())[0, 1])


def _perm_p(ya: np.ndarray, xa: np.ndarray, n_p: int, seed: int) -> tuple[float, float]:
    obs = _corr(ya, xa)
    rng = np.random.default_rng(seed)
    ge = 1
    for _ in range(n_p):
        yp = _y_shuffle(ya, int(rng.integers(0, 2**31 - 2)))
        c = _corr(yp, xa)
        if abs(c) >= abs(obs):
            ge += 1
    return obs, ge / (n_p + 1)


obs, p = _perm_p(y, x, n_perm, info["seed"] + 101)
rc = build_run_card(
    run_id="charter_energy_open_meteo_stream_smoke",
    hypothesis="Open-Meteo tabular stream: index vs temperature_2m; y-shuffle null",
    metric=f"corr_obs={obs:.4f} p_perm={p:.4f} n_perm={n_perm} n={len(y)}",
    null="y_shuffle_temperature_vs_fixed_index",
    truth_scope="HF MWasil/Open_Meteo_weather_dataset_from_2024_to_2025 train stream; methodological sanity only",
    seed=info["seed"],
)
assert_run_card(rc)
(obs, p, len(y))
